In [ ]:
!pip install -q speechbrain jiwer

In [ ]:
import os, glob, csv
import torch, torchaudio
import torch.nn.functional as F

# >>> PUT YOUR KEY HERE (or set it via Colab "Secrets" and os.environ) <<<
GEMINI_API_KEY = "xxx"

ASR_MODEL = "gemini-3.7-flash"
SR = 16000
CHUNK_SEC, CHUNK_HOP = 3.0, 1.5     # multi-chunk embedding windows (50% overlap)
CONF_MARGIN = 0.10                  # min cosine gap for a "confident" assignment
RESIDUAL_CORR = 0.92                # stream~=mixture above this => separation failed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
def load_audio(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    return wav.to(DEVICE)

def save_audio(path, wav):
    wav = wav.detach().cpu()
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    peak = wav.abs().max()
    if peak > 0:
        wav = wav / peak * 0.95
    torchaudio.save(path, wav, SR)


In [ ]:
from speechbrain.inference.separation import SepformerSeparation
from speechbrain.inference.speaker import EncoderClassifier

print("loading separator + speaker encoder (first run downloads weights)...")
_separator = SepformerSeparation.from_hparams(
    source="speechbrain/sepformer-whamr16k",          # native 16 kHz
    savedir="pretrained_models/sepformer-whamr16k",
    run_opts={"device": DEVICE},
)
_spk = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/ecapa",
    run_opts={"device": DEVICE},
)

@torch.no_grad()
def embed(wav):
    """Average L2-normalized ECAPA embeddings over overlapping chunks."""
    T = wav.shape[-1]
    win, hop = int(CHUNK_SEC * SR), int(CHUNK_HOP * SR)
    if T <= win:
        return F.normalize(_spk.encode_batch(wav).squeeze(), dim=0)
    embs = [F.normalize(_spk.encode_batch(wav[..., s:s+win]).squeeze(), dim=0)
            for s in range(0, T - win + 1, hop)]
    return F.normalize(torch.stack(embs).mean(dim=0), dim=0)

def similarity(a, b):
    return torch.dot(embed(a), embed(b)).item()

def correlation(a, b):
    a, b = a.flatten(), b.flatten()
    n = min(a.numel(), b.numel())
    a, b = a[:n] - a[:n].mean(), b[:n] - b[:n].mean()
    return (torch.dot(a, b) / (a.norm() * b.norm()).clamp_min(1e-8)).abs().item()

@torch.no_grad()
def separate(mixture):
    est = _separator.separate_batch(mixture)          # (1, T, 2)
    return [est[0, :, i].unsqueeze(0) for i in range(est.shape[-1])]


In [ ]:
def mode_one_reference(mixture, ref_a, out_dir):
    """One reference -> keep the stream matching it."""
    os.makedirs(out_dir, exist_ok=True)
    streams = separate(mixture)
    sims = [similarity(s, ref_a) for s in streams]
    order = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)
    best, margin = order[0], sims[order[0]] - sims[order[1]]
    print(f"   sims={[round(s,3) for s in sims]} margin={margin:.3f}")
    if margin < CONF_MARGIN:
        print(f"   [WARN] low confidence (margin<{CONF_MARGIN}): both streams "
              f"resemble the reference; output may be unreliable.")
    for i, s in enumerate(streams):
        c = correlation(s, mixture)
        if c >= RESIDUAL_CORR:
            print(f"   [WARN] stream {i} corr w/ mixture={c:.3f}: separation may have failed.")
    out = os.path.join(out_dir, "speaker_A.wav")
    save_audio(out, streams[best])
    print(f"   kept stream {best} -> {out}")
    return {"A": out}

def mode_both_references(mixture, ref_a, ref_b, out_dir):
    """Two references -> assign both streams to the two speakers."""
    os.makedirs(out_dir, exist_ok=True)
    streams = separate(mixture)
    sims = [[similarity(s, r) for r in (ref_a, ref_b)] for s in streams]
    straight, swapped = sims[0][0] + sims[1][1], sims[0][1] + sims[1][0]
    if straight >= swapped:
        assign, margin = {"A": streams[0], "B": streams[1]}, straight - swapped
    else:
        assign, margin = {"A": streams[1], "B": streams[0]}, swapped - straight
    print(f"   matrix={[[round(x,3) for x in row] for row in sims]} margin={margin:.3f}")
    if margin < CONF_MARGIN:
        print(f"   [WARN] ambiguous assignment (margin<{CONF_MARGIN}).")
    for i, s in enumerate(streams):
        c = correlation(s, mixture)
        if c >= RESIDUAL_CORR:
            print(f"   [WARN] stream {i} corr w/ mixture={c:.3f}: separation may have failed.")
    outs = {}
    for spk, wav in assign.items():
        p = os.path.join(out_dir, f"speaker_{spk}.wav")
        save_audio(p, wav)
        outs[spk] = p
    print(f"   wrote {list(outs.values())}")
    return outs


In [ ]:
from google import genai
from google.genai import types

_gemini = genai.Client(api_key=GEMINI_API_KEY)

def transcribe(wav_path, language_hint="Arabic"):
    """Transcribe a WAV file with Gemini. Returns plain transcript text."""
    with open(wav_path, "rb") as f:
        audio_bytes = f.read()
    prompt = (f"Transcribe this {language_hint} audio exactly. "
              f"Output only the transcript text, no commentary, no timestamps.")
    resp = _gemini.models.generate_content(
        model=ASR_MODEL,
        contents=[prompt,
                  types.Part.from_bytes(data=audio_bytes, mime_type="audio/wav")],
    )
    return (resp.text or "").strip()

In [ ]:
def evaluate_wer(manifest_csv, out_root="eval_out", language_hint="Arabic"):
    from jiwer import wer
    rows = list(csv.DictReader(open(manifest_csv)))
    mix_w, sep_w = [], []
    for i, row in enumerate(rows):
        work = os.path.join(out_root, f"row_{i}")
        mixture = load_audio(row["mixture"])
        ref_a = load_audio(row["ref_a"])
        if row.get("ref_b"):
            outs = mode_both_references(mixture, ref_a, load_audio(row["ref_b"]), work)
        else:
            outs = mode_one_reference(mixture, ref_a, work)
        truth = row["truth_a"]
        w_mix = wer(truth, transcribe(row["mixture"], language_hint))
        w_sep = wer(truth, transcribe(outs["A"], language_hint))
        mix_w.append(w_mix); sep_w.append(w_sep)
        print(f"[{i}] WER mixture={w_mix:.3f} separated={w_sep:.3f} delta={w_mix-w_sep:+.3f}")
    n = len(rows)
    print("\n=== WER SUMMARY ===")
    print(f"  files={n}")
    print(f"  mean WER mixture  = {sum(mix_w)/n:.3f}")
    print(f"  mean WER separated= {sum(sep_w)/n:.3f}")
    print(f"  mean improvement  = {(sum(mix_w)-sum(sep_w))/n:+.3f}")
    print("  If improvement is small/negative, this baseline isn't enough ->")
    print("  that's your signal to try a true TSE model (WeSep).")

# evaluate_wer("manifest.csv")

In [ ]:
mixture = load_audio("/content/Recording (44).wav")
ref_a   = load_audio("/content/Recording (52).wav")
ref_b   = load_audio("/content/WhatsApp Ptt 2026-05-24 at 22.25.50.ogg")
outs = mode_both_references(mixture, ref_a, ref_b, "out_both")
print("TRANSCRIPT A:", transcribe(outs["A"]))
print("TRANSCRIPT B:", transcribe(outs["B"]))

In [ ]:
mixture = load_audio("/content/Recording (44).wav")
ref_a   = load_audio("/content/Recording (52).wav")
outs = mode_one_reference(mixture, ref_a, "out_single")
print("TRANSCRIPT A:", transcribe(outs["A"]))

In [ ]:
!nvidia-smi